In [16]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv('all_accelerometer_data_pids_13.csv')

In [3]:
df.head()

,time,pid,x,y,z
0,0,JB3156,0.0000,0.0000,0.0000
1,0,CC6740,0.0000,0.0000,0.0000
2,1493733882409,SA0297,0.0758,0.0273,-0.0102
3,1493733882455,SA0297,-0.0359,0.0794,0.0037
4,1493733882500,SA0297,-0.2427,-0.0861,-0.0163


In [4]:
print(df.shape)

(14057567, 5)


In [5]:
print(df.dtypes)
print(df['pid'].unique())  # pid 이름 확인

time      int64
pid         str
x       float64
y       float64
z       float64
dtype: object
<StringArray>
['JB3156', 'CC6740', 'SA0297', 'PC6771', 'BK7610', 'DC6359', 'MC7070',
 'MJ8002', 'BU4707', 'JR8022', 'HV0618', 'SF3079', 'DK3500']
Length: 13, dtype: str


In [6]:
# TAC 파일 하나 열어서 확인
tac_files = sorted(os.listdir('clean_tac/'))
print(tac_files)  # 파일명 목록

['BK7610_clean_TAC.csv', 'BU4707_clean_TAC.csv', 'CC6740_clean_TAC.csv', 'DC6359_clean_TAC.csv', 'DK3500_clean_TAC.csv', 'HV0618_clean_TAC.csv', 'JB3156_clean_TAC.csv', 'JR8022_clean_TAC.csv', 'MC7070_clean_TAC.csv', 'MJ8002_clean_TAC.csv', 'PC6771_clean_TAC.csv', 'SA0297_clean_TAC.csv', 'SF3079_clean_TAC.csv']


In [7]:
sample = pd.read_csv(f'clean_tac/{tac_files[0]}')
print(sample.head(10))
print(sample.dtypes)
print(sample.shape)

    timestamp  TAC_Reading
0  1493718714    -0.000482
1  1493720697     0.001573
2  1493721027     0.002144
3  1493721357     0.000877
4  1493721686    -0.001145
5  1493722016    -0.002159
6  1493722345    -0.001033
7  1493722674     0.001808
8  1493723003     0.004542
9  1493724832     0.005185
timestamp        int64
TAC_Reading    float64
dtype: object
(57, 2)


In [8]:
# pid별로 묶기
for pid, group in df.groupby('pid'):
    print(f"{pid}: {len(group)} rows")

BK7610: 1225727 rows
BU4707: 447423 rows
CC6740: 2374695 rows
DC6359: 591358 rows
DK3500: 1339622 rows
HV0618: 1876013 rows
JB3156: 1177749 rows
JR8022: 307526 rows
MC7070: 318600 rows
MJ8002: 631303 rows
PC6771: 2141701 rows
SA0297: 962901 rows
SF3079: 662949 rows


In [9]:
all_tac_list = []

for filename in tac_files:
    # Extract PID from filename (e.g., 'BK7610_clean_TAC.csv' -> 'BK7610')
    pid = filename.split('_')[0]
    
    temp_df = pd.read_csv(f'clean_tac/{filename}')
    temp_df['pid'] = pid
    
    # Create the binary label: 1 if TAC > 0.08, else 0
    temp_df['label'] = (temp_df['TAC_Reading'] >= 0.08).astype(int)
    
    # Convert timestamp to milliseconds to match accelerometer 'time'
    # Assuming TAC timestamp is in seconds, we multiply by 1000
    temp_df['time'] = temp_df['timestamp'] * 1000
    
    all_tac_list.append(temp_df)

df_tac = pd.concat(all_tac_list, ignore_index=True)

In [10]:
# 1. Ensure both are sorted by time
df = df.sort_values('time')
df_tac = df_tac.sort_values('time')

# 2. Merge by matching PIDs and finding the nearest previous timestamp
combined_df = pd.merge_asof(
    df, 
    df_tac[['time', 'pid', 'label', 'TAC_Reading']], 
    on='time', 
    by='pid', 
    direction='backward'
)

# 3. Drop any rows where an accelerometer reading happened BEFORE the first TAC reading
combined_df = combined_df.dropna(subset=['label'])

print(combined_df['label'].value_counts())

label
0.0    10708190
1.0     3349375
Name: count, dtype: int64


In [11]:
combined_df.head()

,time,pid,x,y,z,label,TAC_Reading
2,1493733882409,SA0297,0.0758,0.0273,-0.0102,0.0,0.030566
3,1493733882455,SA0297,-0.0359,0.0794,0.0037,0.0,0.030566
4,1493733882500,SA0297,-0.2427,-0.0861,-0.0163,0.0,0.030566
5,1493733883945,SA0297,-0.2888,0.0514,-0.0145,0.0,0.030566
6,1493733883953,SA0297,-0.0413,-0.0184,-0.0105,0.0,0.030566


In [12]:
# Create magnitude column
combined_df['mag'] = (combined_df['x']**2 + combined_df['y']**2 + combined_df['z']**2)**0.5

In [14]:
def extract_window_features(group, window_size=100):
    # In pandas groupby.apply, the grouping key is accessible via group.name
    current_pid = group.name 
    
    features = []
    
    # Ensure there is enough data for at least one window
    if len(group) < window_size:
        return pd.DataFrame()

    for i in range(0, len(group) - window_size, window_size // 2): # 50% overlap
        window = group.iloc[i:i+window_size]
        
        feat = {
            'pid': current_pid,  # Use the stored name here
            'label': window['label'].iloc[0], # Assuming label is constant in window
            'mag_mean': window['mag'].mean(),
            'mag_std': window['mag'].std(),
            'mag_max': window['mag'].max(),
            'mag_var': window['mag'].var(),
            'x_std': window['x'].std(),
            'y_std': window['y'].std(),
            'z_std': window['z'].std(),
        }
        features.append(feat)
        
    return pd.DataFrame(features)

# Apply the function
# Adding include_groups=False is the modern pandas standard to suppress warnings
windowed_data = combined_df.groupby('pid').apply(extract_window_features, include_groups=False).reset_index(drop=True)

print(f"Created {len(windowed_data)} windows across {windowed_data['pid'].nunique()} participants.")
print(windowed_data.head())

Created 281132 windows across 13 participants.
      pid  label  mag_mean   mag_std   mag_max   mag_var     x_std     y_std  \
0  BK7610    0.0  0.072515  0.051678  0.242754  0.002671  0.041507  0.030790   
1  BK7610    0.0  0.042192  0.039420  0.205022  0.001554  0.025541  0.024937   
2  BK7610    0.0  0.080803  0.063333  0.302581  0.004011  0.041169  0.051209   
3  BK7610    0.0  0.075737  0.056229  0.302581  0.003162  0.033942  0.049087   
4  BK7610    0.0  0.033361  0.017356  0.111921  0.000301  0.012130  0.019139   

      z_std  
0  0.059589  
1  0.042858  
2  0.077374  
3  0.065868  
4  0.028466  


In [15]:
X = windowed_data.drop(['label', 'pid'], axis=1)
y = windowed_data['label']
groups = windowed_data['pid']

In [18]:
# 2. Define the Pipeline
# This ensures StandardScaler is fit ONLY on the training groups in each fold.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# 3. Initialize GroupKFold
# Since you have 13 participants, 5 folds is a good balance.
gkf = GroupKFold(n_splits=5)

# 4. Run Cross-Validation
# 'cross_validate' allows us to see multiple metrics (Accuracy, F1, etc.)
metrics = ['accuracy', 'precision', 'recall', 'f1']
results = cross_validate(pipeline, X, y, groups=groups, cv=gkf, scoring=metrics)

# 5. Report Results
print(f"Mean Accuracy: {np.mean(results['test_accuracy']):.3f}")
print(f"Mean F1-Score: {np.mean(results['test_f1']):.3f}")
print(f"Mean Recall (Sensitivity): {np.mean(results['test_recall']):.3f}")

c:\Users\vyrim\OneDrive\Documents\Homework\06Spring2026\4650Repositories\4650FinalProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Mean Accuracy: 0.752
Mean F1-Score: 0.068
Mean Recall (Sensitivity): 0.057
